# Opportunity · Asset Location & Idle Cash

*Location* is the free lunch: the same holdings can owe less tax depending on **which
account** they sit in. Tax-inefficient assets (BDCs, bonds, REITs — ordinary-income
distributions) belong in sheltered accounts; cash should at least be earning a sweep
yield. This audits both.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger, mapping, opportunities as O

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
entries, _, _ = ledger.load()
account_meta = mapping.load_account_meta()

TODAY = pd.Timestamp.today().normalize()
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} · {positions['account_name'].nunique()} accounts · as of {TODAY:%Y-%m-%d}")

## The location map — what sits in which tax bucket

Dollar value by asset class against tax treatment (taxable vs tax-deferred vs tax-free).
The question each row asks: *is this in the right column?*

In [ ]:
lm = O.location_matrix(positions, account_meta)
order = ["taxable", "tax_deferred", "tax_free"]
lm = lm[[c for c in order if c in lm.columns]]
fig = px.imshow(lm, text_auto=".2s", aspect="auto", color_continuous_scale="Blues",
                title="Market value: asset class × tax treatment")
fig.update_traces(hovertemplate="%{y} · %{x}: $%{z:,.0f}<extra></extra>")
fig.update_layout(height=460, coloraxis_showscale=False, xaxis_title="", yaxis_title="")
fig.show()

## Misplaced holdings

Tax-inefficient asset classes (ordinary-income payers) found in **taxable** accounts —
these are the candidates to relocate into an IRA/401k/HSA.

In [ ]:
flags = O.location_flags(positions, account_meta, transactions)
if flags.empty:
    ineff = O.location_matrix(positions, account_meta).reindex(O.TAX_INEFFICIENT).dropna(how="all")
    where = ineff.idxmax(axis=1).to_dict() if not ineff.empty else {}
    print("✓ No tax-inefficient holdings sit in a taxable account.")
    for ac, col in where.items():
        print(f"   {ac}: already in {col}")
else:
    display(flags[["symbol", "account_name", "asset_class", "market_value", "ttm_income"]]
            .style.format({"market_value": "${:,.0f}", "ttm_income": "${:,.0f}"}))

## Cash — earning its keep, or idle?

Cash by account with its trailing-12-month income and **implied yield**. A big balance at
~0% is the opportunity (sweep it into a money-market fund); balances already near the
~4–5% sweep rate are fine.

In [ ]:
ca = O.cash_audit(positions, transactions, account_meta)
fig = px.bar(ca, x="account_name", y="market_value", color="implied_yield",
             color_continuous_scale="RdYlGn", range_color=[0, 0.05],
             title="Cash by account (color = implied yield)",
             hover_data={"ttm_income": ":$,.0f", "implied_yield": ":.2%", "tax_treatment": True})
fig.update_layout(height=420, yaxis_tickprefix="$", coloraxis_colorbar_tickformat=".0%",
                  xaxis_title="", yaxis_title="cash")
fig.show()

idle = ca[(ca["implied_yield"] < 0.01) & (ca["market_value"] > 5_000)]
total_cash = ca["market_value"].sum()
print(f"Total cash: ${total_cash:,.0f} ({total_cash/TOTAL:.0%} of portfolio)")
if idle.empty:
    print("✓ No large balance is sitting idle — cash is earning a sweep yield.")
else:
    sweep = 0.043
    upside = (idle["market_value"] * (sweep - idle["implied_yield"].fillna(0))).sum()
    print(f"⚠ ${idle['market_value'].sum():,.0f} earning <1% — at a {sweep:.1%} sweep that's "
          f"~${upside:,.0f}/yr left on the table:")
    display(idle[["account_name", "market_value", "implied_yield"]])

## Allocation lens — cash as opportunity cost

Beyond *idle* cash, the **size** of the cash sleeve is a choice. Here's what the current
cash earned vs. what the broad market returned over the window — the gap is the risk
premium you're forgoing for liquidity/safety (a preference, not a free lunch).

In [ ]:
spy_cagr = (history["SPY"].iloc[-1] / history["SPY"].iloc[0]) ** (252 / len(history)) - 1
mmf = ca["implied_yield"].replace(0, np.nan).median()
print(f"Cash sleeve: ${total_cash:,.0f}  ·  earning ~{mmf:.1%} (median sweep)")
print(f"Broad market (SPY) over the window: ~{spy_cagr:.1%} annualized")
print(f"Forgone premium on the cash sleeve: ~${total_cash * (spy_cagr - mmf):,.0f}/yr "
      f"— for materially more risk. A liquidity/allocation decision, not a leak.")

---
*These are **candidate generators**, not advice — every row is a starting point for your
own judgment (and, for anything tax-related, a CPA's). Prices are research-grade
(yfinance); cost basis and lots come from the curated ledger, so they're only as right as
its fixups.*